In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/hertie-dsa-26/project-scream/refs/heads/main/literature/diabetes_risk_literature_subset.csv"
df = pd.read_csv(url)

In [2]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# ── 2. LOAD & CLEAN DATA ──────────────────────────────────────────────────────
# Strip whitespace from column names (raw data has leading spaces)
df.columns = df.columns.str.strip()

In [11]:
# ── 3. FEATURE SELECTION ──────────────────────────────────────────────────────
# Drop leakage columns: 'has_diabetes' (text version of target) and 'bmi_x100' (duplicate)
dfs = df.drop(columns=['has_diabetes', 'bmi_x100'])

features = [
    'general_health',
    'any_physical_activity',
    'sex',
    'age_imputed',
    'bmi',                   # raw BMI
    'education_level',
    'income_level',
    'smoking_status',
    'any_alcohol_past_30d'
]

X = dfs[features]
y = dfs['has_diabetes_binary']           # 0 = No Diabetes, 1 = Diabetes

# One-hot encode categoricals including bmi_category
X = pd.get_dummies(X, drop_first=True)

# Save feature columns BEFORE train_test_split (needed for inference later)
feature_columns = X.columns.tolist()
with open("feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

In [12]:
# ── 4. TRAIN/TEST SPLIT ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify preserves class ratio
)

In [13]:
# ── 5. SCALING ────────────────────────────────────────────────────────────────
# Fit scaler on train only — transform both train and test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [14]:
# ── 6. MODEL DEFINITION ───────────────────────────────────────────────────────
class ManualSVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=100, class_weight=None):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.class_weight = class_weight
        self.w = None
        self.b = None

    def _compute_class_weights(self, y_internal):
        """Balanced weighting on ±1 labels: n_samples / (n_classes * class_count)."""
        classes = np.unique(y_internal)
        n_samples = len(y_internal)
        return {c: n_samples / (len(classes) * np.sum(y_internal == c)) for c in classes}

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        # Convert 0/1 → ±1 internally for hinge loss; y stays 0/1 outside
        y_ = np.where(y == 0, -1, 1)

        if self.class_weight == 'balanced':
            cw = self._compute_class_weights(y_)
        elif isinstance(self.class_weight, dict):
            cw = self.class_weight
        else:
            cw = {c: 1.0 for c in np.unique(y_)}

        sample_weights = np.array([cw[label] for label in y_])

        for _ in range(self.n_iters):
            for i in range(n_samples):
                condition = y_[i] * (np.dot(X[i], self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (
                        2 * self.lambda_param * self.w - sample_weights[i] * np.dot(X[i], y_[i])
                    )
                    self.b -= self.lr * sample_weights[i] * y_[i]

    def predict(self, X):
        linear_output = np.dot(X, self.w) - self.b
        return (linear_output >= 0).astype(int)      # 0 = No Diabetes, 1 = Diabetes

In [15]:
# ── 7. TRAIN ──────────────────────────────────────────────────────────────────
model = ManualSVM(lr=0.001, lambda_param=0.01, n_iters=300, class_weight={-1: 1.0, 1: 5.0})
model.fit(X_train, y_train.values)

In [16]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, recall_score, precision_score
)

# ── 8. EVALUATE ───────────────────────────────────────────────────────────────
predictions = model.predict(X_test)

print("=== Manual SVM ===")
print(f"Accuracy  : {np.mean(predictions == y_test.values):.4f}")
print(f"Recall    : {recall_score(y_test, predictions):.4f}")
print(f"Precision : {precision_score(y_test, predictions):.4f}")
print(f"F1 Score  : {f1_score(y_test, predictions):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, predictions):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['No Diabetes', 'Diabetes']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))


=== Manual SVM ===
Accuracy  : 0.7103
Recall    : 0.7059
Precision : 0.3038
F1 Score  : 0.4248
ROC-AUC   : 0.7085

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.93      0.71      0.81     77487
    Diabetes       0.30      0.71      0.42     13841

    accuracy                           0.71     91328
   macro avg       0.62      0.71      0.62     91328
weighted avg       0.84      0.71      0.75     91328

Confusion Matrix:
[[55101 22386]
 [ 4071  9770]]


In [20]:
# ── 9. SAVE PIPELINE ──────────────────────────────────────────────────────────
joblib.dump(model, "svm_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved.")

Model and scaler saved.


In [18]:
# ── 10. INFERENCE CHECK (single sample) ───────────────────────────────────────
sample_scaled = X_test[[0]]             # already scaled — no need to re-transform
pred = model.predict(sample_scaled)
print("Sample prediction:", "Diabetes" if pred[0] == 1 else "No Diabetes")

Sample prediction: Diabetes
